In [28]:
################ Import necessary libraries

%pip install pyblp
%pip install statsmodels
%pip install linearmodels
%pip install stargazer

import pandas as pd
import numpy as np
import pyblp 
import statsmodels.api as sm_api
from statsmodels.sandbox.regression.gmm import IV2SLS
from linearmodels import IV2SLS
from stargazer.stargazer import Stargazer
import matplotlib.pyplot as plt

pyblp.options.digits = 2
pyblp.options.verbose = False
pyblp.__version__

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


'1.1.2'

In [244]:
################ Load data ready for nested logit
df = pd.read_csv('data_with_IV.csv')

# Generate nesting_id
df['nesting_ids'] = pd.factorize(df['fuel_type'])[0]

# Generate log of charging station stock
df['log_charging_IV'] = np.log(df['charging_stations_stock_lag']) 

# Create indicator for electric vehicles
df['is_electric'] = df['type'].apply(lambda x: 1 if x == '国产新能源乘用车' else 0)

# Make year an object variable
df['year'] = df['year'].astype(str)

# Rename columns to match pyblp requirements
df.rename(columns={
    'product_id': 'product_ids',
    'province_id': 'province_ids',
    'market_id': 'market_ids',
    'weighted_Avg_Price': 'prices',
    'market_share': 'shares'
}, inplace=True)

# calculate number of products in each market x nest
df['product_set_size'] = df.groupby(['market_ids', 'nesting_ids'])['product_ids'].transform('count')

In [245]:
# 定义工具变量列表
iv_list = [
    'cost_shifter', 
    'product_set_size',
    'euclidean_range',
    'local_range',
    'euclidean_battery',
    'local_battery',
    'euclidean_power',
    'local_power'
]

# 拼接成公式字符串
iv_str = ' + '.join(iv_list)


In [246]:
# Check for variations in number of available products over time and across provinces

# 统计每个省份每年可用产品数
product_counts = df.groupby(['province_ids', 'year'])['product_ids'].nunique().reset_index(name='num_products')
print(product_counts)

# 检查每年不同省份的产品数分布
print("\n产品数在每年各省的分布：")
print(product_counts.groupby('year')['num_products'].describe())

# 检查每个省份不同年份的产品数分布
print("\n产品数在各省每年的分布：")
print(product_counts.groupby('province_ids')['num_products'].describe())

    province_ids  year  num_products
0            P01  2019           114
1            P01  2020           146
2            P01  2021           187
3            P01  2022           184
4            P01  2023           228
..           ...   ...           ...
150          P31  2019           107
151          P31  2020           134
152          P31  2021           174
153          P31  2022           172
154          P31  2023           212

[155 rows x 3 columns]

产品数在每年各省的分布：
      count        mean        std    min    25%    50%    75%    max
year                                                                 
2019   31.0  114.387097   6.591294   96.0  111.5  116.0  118.0  123.0
2020   31.0  150.677419  12.343384  117.0  144.0  155.0  160.0  163.0
2021   31.0  188.000000  13.271523  151.0  184.0  193.0  197.5  204.0
2022   31.0  180.193548  12.666016  143.0  175.5  185.0  188.5  193.0
2023   31.0  240.838710  22.737776  162.0  230.5  249.0  255.5  266.0

产品数在各省每年的分布：
              

In [253]:
################ 2SLS model using custom instruments

# 计算 log(sjm), log(s0m), log(sj/g)
df['log_sjm'] = np.log(df['shares'])
df['log_s0m'] = np.log(1 - df.groupby('market_ids')['shares'].transform('sum'))
df['log_sj_g'] = np.log(df['shares'] / df.groupby(['market_ids', 'nesting_ids'])['shares'].transform('sum'))
df['log_charging_stock'] = np.log(df['charging_stations_stock'])
df['Intercept'] = 1


# Define the formula for the IV2SLS model
formula = f'''
(log_sjm - log_s0m) ~ 0 + is_electric*log_charging_IV + range + power + battery_capacity
    + [prices + log_sj_g ~ {iv_str}]
'''

iv_model = IV2SLS.from_formula(formula, data=df).fit(cov_type="clustered", clusters=df['market_ids'])

# 
main_vars = ['Intercept', 'prices', 'power', 'range', 'battery_capacity', 'log_charging_IV', 'log_sj_g']

print(iv_model.summary)

                          IV-2SLS Estimation Summary                          
Dep. Variable:                log_sjm   R-squared:                      0.9880
Estimator:                    IV-2SLS   Adj. R-squared:                 0.9880
No. Observations:               33545   F-statistic:                 7.627e+04
Date:                Sun, Jul 27 2025   P-value (F-stat)                0.0000
Time:                        02:35:01   Distribution:                  chi2(8)
Cov. Estimator:             clustered                                         
                                                                              
                                      Parameter Estimates                                      
                             Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
-----------------------------------------------------------------------------------------------
is_electric                    -12.584     0.6422    -19.596     0.0000     -13.

In [250]:
# Check first stage results
print(iv_model.first_stage.summary)

             First Stage Estimation Results            
                                    prices     log_sj_g
-------------------------------------------------------
R-squared                           0.5179       0.0891
Partial R-squared                   0.1164       0.0078
Shea's R-squared                    0.1071       0.0071
Partial F-statistic                 1777.3       142.18
P-value (Partial F-stat)            0.0000       0.0000
Partial F-stat Distn               chi2(8)      chi2(8)
============================= ============ ============
is_electric                        -2.2065       1.1613
                                 (-2.2381)     (4.0015)
log_charging_IV                    -0.2777      -0.3125
                                 (-1.3915)    (-9.5661)
range                               0.0123      -0.0013
                                  (20.569)    (-9.6018)
power                               0.1256       0.0047
                                  (36.730)     (

In [260]:
################ Load supply-side data
supply_df = pd.read_csv('supply_side_data.csv')

In [261]:
supply_df.head()

,province,year,charging_stations_stock,EV_stock,road_length,yearly_gas_price_avg,shift_share_IV,road_fuel_IV,num_models_in_market,sales_weighted_avg_range
0,上海市,2019,55113.0,304602,13106.0,7023.074286,3.126124e+10,9.204441e+07,123.0,329.518520
1,上海市,2020,85538.0,425193,13045.0,5803.269444,4.584036e+10,7.570365e+07,149.0,298.620943
2,上海市,2021,103249.0,665472,12917.0,7802.940278,5.800445e+10,1.007906e+08,186.0,306.336070
3,上海市,2022,122235.0,998341,13082.0,9194.876389,9.391351e+10,1.202874e+08,196.0,334.666560
4,上海市,2023,147708.0,1171415,13005.0,9157.885714,1.586627e+11,1.190983e+08,212.0,414.243129


In [267]:
################ Supply side model

# Create a time trend variable
supply_df['time_trend'] = supply_df['year'].astype(int) - supply_df['year'].astype(int).min() + 1

# Add time_trend
formula = 'log(charging_stations_stock) ~ [log(EV_stock) ~ road_fuel_IV + num_models_in_market + sales_weighted_avg_range] + C(province) + time_trend'

supply_model = IV2SLS.from_formula(formula, data=supply_df).fit(
)
print(supply_model.summary)
print(supply_model.first_stage.summary)

                               IV-2SLS Estimation Summary                               
Dep. Variable:     log(charging_stations_stock)   R-squared:                      0.9734
Estimator:                              IV-2SLS   Adj. R-squared:                 0.9665
No. Observations:                           155   F-statistic:                 9.135e+05
Date:                          Sun, Jul 27 2025   P-value (F-stat)                0.0000
Time:                                  02:51:58   Distribution:                 chi2(33)
Cov. Estimator:                          robust                                         
                                                                                        
                                   Parameter Estimates                                   
                       Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
-----------------------------------------------------------------------------------------
C(province)[上海市]  

In [ ]:
# Export model summary as LaTeX
# Stargazer does not support different covariate orders for each model,
# so the best practice is to include all you want to show and accept blanks for the other.
main_vars = ['Intercept', 'log(EV_stock)', 'time_trend', 'shift_share_IV']

stargazer = Stargazer([model, first_stage])
stargazer.covariate_order(main_vars)

with open('supply_and_first_stage_stargazer.tex', 'w', encoding='utf-8') as f:
    f.write(stargazer.render_latex())

